# Multiaxial Fatigue Life Prediction: XGBoost & SHAP Analysis

**Makale:** Metalik Malzemelerde Çok Eksenli Yorulma Ömrünün Tahmininde XGBoost ve SHAP Analizi  
**Yazar:** [Yazar Adı Soyadı]  
**Dergi:** [Dergi Adı], 2025  

---

## Çalışma Hakkında

Bu notebook, 40 metalik malzemeye ait 1.167 çok eksenli yorulma örneği üzerinde:
- **XGBoost** regresyon modelinin eğitimini ve değerlendirmesini,
- **SWT** klasik kriteri ile karşılaştırmayı,
- **SHAP** analizi ile özellik öneminin hesaplanmasını,
- Yükleme yolu kategorilerine göre **stratified** istatistiksel karşılaştırmayı

içermektedir.

## Veri Seti

Chen et al. (2024), *Scientific Data*, 11:1027  
DOI: [10.1038/s41597-024-03862-4](https://doi.org/10.1038/s41597-024-03862-4)

## Metodolojik Notlar

| # | Konu | Uygulama |
|---|------|----------|
| 1 | Hiperparametre seçimi | Manuel iteratif arama; test seti hiçbir aşamada kullanılmadı |
| 2 | Train/test bölümü | LP kategorisine göre **stratified split** (%80/%20) |
| 3 | SHAP hesabı | Yalnızca **test setine** uygulandı (eğitim verisi dahil değil) |
| 4 | Bootstrap CI | Test setine, n_boot=1000, %95 güven düzeyi |
| 5 | İstatistiksel test | Bonferroni düzeltmeli Mann-Whitney U (α=0.05/12=0.0042) |
| 6 | Kararlılık kontrolü | 5×10 Tekrarlı K-Fold CV |
| 7 | SWT karşılaştırması | Kalibre edilmemiş formülasyon (makale Bölüm 3.2'de açıklandı) |

## Kurulum

```bash
pip install xgboost shap scikit-learn pandas numpy matplotlib scipy
```

## Kullanım

Google Colab'da **Hücre 1**'deki yol değişkenlerini ayarlayın, ardından tüm hücreleri sırayla çalıştırın.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 1 — Kurulum, Bağımlılıklar ve Drive Bağlantısı           ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import drive
drive.mount('/content/drive')

import os, warnings, subprocess
warnings.filterwarnings('ignore')
subprocess.run(['pip', 'install', 'shap', 'xgboost', '-q'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

import shap
import xgboost as xgb
from scipy.stats import mannwhitneyu

from sklearn.model_selection import (
    train_test_split, cross_val_score, RepeatedKFold
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor

# ── Yolları buradan ayarlayın ──────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive'
OUT          = os.path.join(DRIVE_ROOT, 'makale_v2')
DATASET_BASE = '/content/dataset/A Deep Learning Dataset for Metal Multiaxial Fatigue Life Prediction'
MASTER_CSV   = os.path.join(DRIVE_ROOT, 'master_dataset_fixed.csv')
# ───────────────────────────────────────────────────────────────────

os.makedirs(OUT, exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Kurulum tamamlandı")
print(f"  XGBoost : {xgb.__version__}")
print(f"  SHAP    : {shap.__version__}")
print(f"  Çıktı   : {OUT}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 2 — Ham Veri Yükleme                                     ║
# ╚══════════════════════════════════════════════════════════════════╝

df = pd.read_csv(MASTER_CSV)

print(f"Toplam örnek      : {len(df)}")
print(f"Sütunlar          : {list(df.columns)}")
print(f"\nLP Kategori dağılımı:\n{df['lp_category'].value_counts()}")
print(f"\nMalzeme dağılımı:\n{df['mat_category'].value_counts()}")
print(f"\nlgNf özeti:\n{df['lgNf'].describe().round(3)}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 3 — Zaman Serisinden Fiziksel Özellik Çıkarımı           ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Çıkarılan özellikler:
#   sigma_a     : Eksenel gerilme genliği       (tepe-vadi / 2)
#   sigma_mean  : Ortalama eksenel yükleme
#   R           : Yük oranı                      (sigma_min / sigma_max)
#   tau_a       : Kayma gerilmesi genliği        (tepe-vadi / 2)
#   lambda      : Gerilme oranı                  (tau_a / sigma_a)
#   phase_proxy : Faz açısı vekili               (Pearson korelasyonu)
#
# NOT: phase_proxy, eksenel ve kayma kanalları arasındaki Pearson
# korelasyonudur. Bu ölçüt, sinüzoidal yüklemede faz açısıyla
# monoton ilişkilidir (φ=0° → r=1, φ=90° → r≈0). Düzensiz dalga
# formlarında yaklaşık kalır; bu sınırlılık makale Bölüm 4.4'te
# tartışılmıştır.

def extract_features(filepath: str) -> dict | None:
    try:
        raw  = pd.read_csv(filepath, header=None, encoding='latin1')
        col1 = pd.to_numeric(raw.iloc[:, 0], errors='coerce').dropna().values
        col2 = pd.to_numeric(raw.iloc[:, 1], errors='coerce').dropna().values
        if len(col1) < 10:
            return None
        sigma_a     = (col1.max() - col1.min()) / 2
        sigma_mean  = col1.mean()
        sigma_max   = col1.max()
        sigma_min   = col1.min()
        R           = sigma_min / sigma_max if abs(sigma_max) > 1e-10 else -1.0
        tau_a       = (col2.max() - col2.min()) / 2 if len(col2) > 10 else 0.0
        lam         = tau_a / sigma_a if abs(sigma_a) > 1e-10 else 0.0
        phase_proxy = (
            float(np.corrcoef(col1, col2)[0, 1])
            if (len(col1) == len(col2) and len(col1) > 2) else 0.0
        )
        return {'sigma_a': sigma_a, 'sigma_mean': sigma_mean, 'R': R,
                'tau_a': tau_a, 'lambda': lam, 'phase_proxy': phase_proxy}
    except Exception:
        return None


strain_folder = os.path.join(DATASET_BASE, 'All data_Strain')
stress_folder = os.path.join(DATASET_BASE, 'All data_Stress')
results, skipped = [], 0

print("Özellik çıkarımı başlıyor (1167 dosya)…")
for _, row in df.iterrows():
    fname  = str(row['filename']).strip()
    folder = strain_folder if row['control'] == 'strain' else stress_folder
    feats  = extract_features(os.path.join(folder, fname))
    if feats:
        entry = {k: row[k] for k in
                 ['E_GPa','UTS_MPa','Sy_MPa','nu',
                  'lp_category','lp_code','mat_category','lgNf','filename']}
        entry.update(feats)
        results.append(entry)
    else:
        skipped += 1

master = pd.DataFrame(results)
master.to_csv(os.path.join(OUT, 'master_with_features.csv'), index=False)
print(f"✓ Tamamlandı: {len(master)} örnek, {skipped} atlandı")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 4 — Veri Seti Özet Grafikleri  (Şekil 1)                 ║
# ╚══════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

master['lgNf'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('lgNf Dağılımı', fontsize=12)
axes[0].set_xlabel('lg(Nf)'); axes[0].set_ylabel('Frekans')

lp_counts = master['lp_category'].value_counts()
bars = axes[1].bar(lp_counts.index, lp_counts.values,
                   color=['#2196F3','#4CAF50','#FF5722'])
axes[1].set_title('Yükleme Yolu Kategorisi', fontsize=12)
for bar, val in zip(bars, lp_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 5, str(val),
                 ha='center', va='bottom', fontsize=10)

mat_counts = master['mat_category'].value_counts()
axes[2].barh(mat_counts.index, mat_counts.values, color='seagreen')
axes[2].set_title('Malzeme Kategorisi', fontsize=12)
axes[2].set_xlabel('Örnek Sayısı')

plt.tight_layout()
plt.savefig(os.path.join(OUT, '01_veri_ozet.png'), bbox_inches='tight', dpi=150)
plt.show()
print("✓ Kaydedildi: 01_veri_ozet.png")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 5 — Model Hazırlığı ve Stratified Train/Test Split       ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Hiperparametre notu:
#   n_estimators=400, learning_rate=0.04, max_depth=5 değerleri
#   eğitim seti üzerinde 5-katlı CV R² izlenerek iteratif manuel
#   arama yoluyla belirlenmiştir. Test seti hiperparametre seçiminin
#   hiçbir aşamasında kullanılmamıştır.
#
# Stratified split notu:
#   Veri seti LP kategorileri arasında dengesizdir
#   (826 Uniaxial / 191 Proportional / 150 Nonproportional).
#   Test setinde kategori oranlarını korumak için lp_encoded
#   değişkenine göre katmanlı örnekleme uygulanmaktadır.

FEAT_COLS = ['E_GPa','UTS_MPa','Sy_MPa','nu',
             'sigma_a','tau_a','lambda','R','phase_proxy']

# LP kategorisi → sayısal kodlama
# Alfabetik sıra: 0=Nonproportional, 1=Proportional, 2=Uniaxial
le_lp = LabelEncoder()
master['lp_encoded'] = le_lp.fit_transform(master['lp_category'])
print("LP encoding:", dict(zip(le_lp.classes_, le_lp.transform(le_lp.classes_))))

ALL_FEAT = FEAT_COLS + ['lp_encoded']

for col in ALL_FEAT + ['lgNf']:
    master[col] = pd.to_numeric(master[col], errors='coerce')

df_model = master[ALL_FEAT + ['lgNf','lp_category']].dropna().copy().reset_index(drop=True)
X        = df_model[ALL_FEAT].values
y        = df_model['lgNf'].values
lp_label = df_model['lp_category'].values   # bootstrap / istatistiksel test için
lp_strat = df_model['lp_encoded'].values     # stratify argümanı için

# ── Stratified split ────────────────────────────────────────────────
X_train, X_test, y_train, y_test, lp_train, lp_test = train_test_split(
    X, y, lp_label,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=lp_strat          # ← sınıf oranlarını korur
)

print(f"\nEğitim : {len(X_train)} örnek")
print(f"Test   : {len(X_test)} örnek")
print("\nTest seti LP dağılımı (stratified doğrulama):")
for cat in ['Uniaxial','Proportional','Nonproportional']:
    n_tot  = (lp_label == cat).sum()
    n_test = (lp_test  == cat).sum()
    print(f"  {cat:<18}: test={n_test:>3}  "
          f"(beklenen≈{int(n_tot*0.2):>3}, oran={n_test/len(lp_test):.1%})")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 6 — Model Eğitimi, Karşılaştırma ve Kararlılık Testi    ║
# ╚══════════════════════════════════════════════════════════════════╝

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name):
    """
    Modeli eğitir, test setinde değerlendirir, 5-katlı CV uygular.
    Döndürür: (metrik_dict, y_pred_array)
    """
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    rmse   = np.sqrt(mean_squared_error(y_te, y_pred))
    mae    = mean_absolute_error(y_te, y_pred)
    r2     = r2_score(y_te, y_pred)
    ratio  = 10**y_pred / 10**y_te
    sb2x   = np.mean((ratio <= 2) & (ratio >= 0.5)) * 100
    sb3x   = np.mean((ratio <= 3) & (ratio >= 1/3)) * 100
    cv     = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2')
    return {
        'Model':      name,
        'R2_test':    round(r2, 4),
        'RMSE':       round(rmse, 4),
        'MAE':        round(mae, 4),
        'SB_2x_%':    round(sb2x, 1),
        'SB_3x_%':    round(sb3x, 1),
        'CV_R2_mean': round(cv.mean(), 4),
        'CV_R2_std':  round(cv.std(), 4),
    }, y_pred


models = {
    'Ridge (baseline)': Ridge(alpha=1.0),
    'Random Forest':    RandomForestRegressor(
                            n_estimators=200, max_depth=8,
                            random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':          xgb.XGBRegressor(
                            n_estimators=400, learning_rate=0.04,
                            max_depth=5, subsample=0.8,
                            colsample_bytree=0.8,
                            random_state=RANDOM_STATE, verbosity=0),
}

perf_list, preds_dict = [], {}
for name, model in models.items():
    print(f"  Eğitiliyor: {name}…")
    perf, y_pred = evaluate_model(model, X_train, y_train, X_test, y_test, name)
    perf_list.append(perf)
    preds_dict[name] = y_pred
    print(f"    R²={perf['R2_test']:.4f}  RMSE={perf['RMSE']:.4f}  "
          f"±3x={perf['SB_3x_%']}%  CV={perf['CV_R2_mean']:.4f}±{perf['CV_R2_std']:.4f}")

perf_df = pd.DataFrame(perf_list)
perf_df.to_csv(os.path.join(OUT, '02_model_karsilastirma.csv'), index=False)
print("\n── Model Karşılaştırması ─────────────────────────────────────")
print(perf_df.to_string(index=False))

# ── 5×10 Tekrarlı K-Fold — Tek Split Kararlılığını Doğrulama ──────
# Tek bir random_state=42 bölümünün sonuçlarının şansa bağlı
# varyansını ölçmek için tekrarlı çapraz doğrulama uygulanmaktadır.
print("\n── 5×10 Tekrarlı K-Fold Kararlılık Testi (XGBoost) ──────────")
xgb_stab = xgb.XGBRegressor(n_estimators=400, learning_rate=0.04,
                              max_depth=5, subsample=0.8,
                              colsample_bytree=0.8,
                              random_state=RANDOM_STATE, verbosity=0)
rkf       = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
rkf_scores = cross_val_score(xgb_stab, X, y, cv=rkf, scoring='r2')
print(f"  5×10 Tekrarlı CV  R² = {rkf_scores.mean():.4f} ± {rkf_scores.std():.4f}")
print(f"  Tek bölüm test    R² = {perf_list[2]['R2_test']:.4f}")
print(f"  → Mutlak fark       : {abs(rkf_scores.mean()-perf_list[2]['R2_test']):.4f}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 7 — Scatter Grafikleri — 3 Model  (Şekil 2)              ║
# ╚══════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {'Ridge (baseline)':'#78909C',
          'Random Forest':   '#43A047',
          'XGBoost':         '#1E88E5'}

for ax, (name, y_pred) in zip(axes, preds_dict.items()):
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    lims = [min(y_test.min(), y_pred.min())-0.3,
            max(y_test.max(), y_pred.max())+0.3]
    ax.scatter(y_test, y_pred, alpha=0.45, s=18, color=colors[name])
    ax.plot(lims, lims, 'k-', lw=1)
    ax.fill_between(lims, [l-np.log10(3) for l in lims],
                    [l+np.log10(3) for l in lims],
                    alpha=0.08, color='green', label='±3x')
    ax.fill_between(lims, [l-np.log10(2) for l in lims],
                    [l+np.log10(2) for l in lims],
                    alpha=0.12, color='orange', label='±2x')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Gerçek lg(Nf)')
    ax.set_ylabel('Tahmin lg(Nf)')
    ax.set_title(f'{name}\nR²={r2:.3f}  RMSE={rmse:.3f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUT, '03_scatter_modeller.png'), bbox_inches='tight', dpi=150)
plt.show()
print("✓ Kaydedildi: 03_scatter_modeller.png")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 8 — SWT Klasik Kriteri vs XGBoost  (Şekil 3, Tablo 2)   ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# SWT formülü: P_SWT = sqrt(sigma_max × sigma_a × E)
#
# Buradaki formülasyon malzemeye özgü kalibrasyon katsayısı
# kullanmayan "kalibre edilmemiş" yaklaşımdır. 40 farklı malzemeyi
# kapsayan heterojen veri setinde her malzeme için ayrı kalibrasyon
# yapmanın pratik güçlüğü nedeniyle bu tercih yapılmıştır.
# Makale Bölüm 3.2'de bu sınırlılık açıkça belirtilmiştir.

master_feat = pd.read_csv(os.path.join(OUT, 'master_with_features.csv'))
for col in ['E_GPa','sigma_a','sigma_mean']:
    master_feat[col] = pd.to_numeric(master_feat[col], errors='coerce')

master_feat['sigma_max']    = (master_feat['sigma_mean'] + master_feat['sigma_a']).clip(lower=1e-3)
master_feat['sigma_a_clip'] = master_feat['sigma_a'].clip(lower=1e-3)
master_feat['P_SWT']        = np.sqrt(
    master_feat['sigma_max'] * master_feat['sigma_a_clip'] * master_feat['E_GPa'])
master_feat['logP_SWT']     = np.log10(master_feat['P_SWT'].clip(lower=1e-10))

df_swt = master_feat[['logP_SWT','lgNf']].replace([np.inf,-np.inf], np.nan).dropna()
X_swt  = df_swt['logP_SWT'].values.reshape(-1,1)
y_swt  = df_swt['lgNf'].values

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_swt, y_swt, test_size=0.2, random_state=RANDOM_STATE)
swt_model  = LinearRegression().fit(X_tr_s, y_tr_s)
y_pred_swt = swt_model.predict(X_te_s)

r2_swt   = r2_score(y_te_s, y_pred_swt)
rmse_swt = np.sqrt(mean_squared_error(y_te_s, y_pred_swt))
mae_swt  = mean_absolute_error(y_te_s, y_pred_swt)
ratio_s  = 10**y_pred_swt / 10**y_te_s
sb2_swt  = np.mean((ratio_s<=2)&(ratio_s>=0.5))*100
sb3_swt  = np.mean((ratio_s<=3)&(ratio_s>=1/3))*100

# XGBoost final model (SHAP için de kullanılacak)
xgb_model = xgb.XGBRegressor(n_estimators=400, learning_rate=0.04,
                               max_depth=5, subsample=0.8, colsample_bytree=0.8,
                               random_state=RANDOM_STATE, verbosity=0)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

r2_xgb   = r2_score(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
ratio_x  = 10**y_pred_xgb / 10**y_test
sb2_xgb  = np.mean((ratio_x<=2)&(ratio_x>=0.5))*100
sb3_xgb  = np.mean((ratio_x<=3)&(ratio_x>=1/3))*100

compare_df = pd.DataFrame([
    {'Model':'SWT (Klasik — kalibre edilmemiş)',
     'R2':round(r2_swt,4),'RMSE':round(rmse_swt,4),'MAE':round(mae_swt,4),
     'SB_2x_%':round(sb2_swt,1),'SB_3x_%':round(sb3_swt,1)},
    {'Model':'XGBoost (Bu Çalışma)',
     'R2':round(r2_xgb,4),'RMSE':round(rmse_xgb,4),'MAE':round(mae_xgb,4),
     'SB_2x_%':round(sb2_xgb,1),'SB_3x_%':round(sb3_xgb,1)},
])
compare_df.to_csv(os.path.join(OUT,'12_swt_vs_xgboost.csv'), index=False)
print(compare_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (name, yp, yt, color) in zip(axes, [
    ('SWT Klasik Kriteri',    y_pred_swt, y_te_s, '#E53935'),
    ('XGBoost (Bu Çalışma)', y_pred_xgb, y_test,  '#1E88E5'),
]):
    r2_p   = r2_score(yt, yp)
    rmse_p = np.sqrt(mean_squared_error(yt, yp))
    sb3_p  = np.mean((10**yp/10**yt<=3)&(10**yp/10**yt>=1/3))*100
    lims   = [min(yt.min(),yp.min())-0.3, max(yt.max(),yp.max())+0.3]
    ax.scatter(yt, yp, alpha=0.45, s=20, color=color)
    ax.plot(lims, lims, 'k-', lw=1.2, label='1:1')
    ax.fill_between(lims,[l-np.log10(3) for l in lims],[l+np.log10(3) for l in lims],
                    alpha=0.08, color='green', label='3x')
    ax.fill_between(lims,[l-np.log10(2) for l in lims],[l+np.log10(2) for l in lims],
                    alpha=0.13, color='orange', label='2x')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Gerçek lg(Nf)', fontsize=11)
    ax.set_ylabel('Tahmin lg(Nf)', fontsize=11)
    ax.set_title(f'{name}\nR²={r2_p:.3f}  RMSE={rmse_p:.3f}  ±3x={sb3_p:.0f}%', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Klasik Model (SWT) vs Makine Öğrenmesi (XGBoost)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT,'13_swt_vs_xgboost.png'), bbox_inches='tight', dpi=150)
plt.show()
print("✓ Kaydedildi: 13_swt_vs_xgboost.png, 12_swt_vs_xgboost.csv")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 9 — Global SHAP Analizi  (Şekil 4, Tablo 3)             ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# SHAP değerleri YALNIZCA TEST SETİNE hesaplanmaktadır.
# Eğitim örnekleri SHAP analizine dahil edilmemiştir; bu sayede
# model çıktıları üzerindeki özellik etkileri, modelin hiç
# görmediği örnekler üzerinden tarafsız biçimde ölçülmektedir.

feature_names = ALL_FEAT

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)   # ← SADECE test seti

# Global özellik önemi — ortalama mutlak SHAP değeri
mean_shap = np.abs(shap_values).mean(axis=0)
importance_df = (
    pd.DataFrame({'feature': feature_names, 'mean_abs_shap': mean_shap})
    .sort_values('mean_abs_shap', ascending=False)
    .reset_index(drop=True)
)
importance_df['rank'] = range(1, len(importance_df)+1)
importance_df.to_csv(os.path.join(OUT,'05_shap_importance.csv'), index=False)

print("── Global SHAP Özellik Önemi ─────────────────────────────────")
print(importance_df[['rank','feature','mean_abs_shap']].to_string(index=False))

# SHAP ham değerlerini kaydet
pd.DataFrame(shap_values, columns=feature_names).to_csv(
    os.path.join(OUT,'04_shap_values_test.csv'), index=False)

# Summary plot (Şekil 4)
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                  show=False, plot_type='dot')
plt.title('SHAP Global Summary — Test Seti', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT,'06_shap_global.png'), bbox_inches='tight', dpi=150)
plt.show()
print("✓ Kaydedildi: 06_shap_global.png")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 10 — Bootstrap CI + Mann-Whitney Testi  (Şekil 5)        ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# [A] Bootstrap güven aralıkları
#     - Test setine uygulanır (n_boot=1000, %95 CI)
#     - Her LP kategorisi kendi test alt kümesi üzerinde hesaplanır
#
# [B] Bonferroni düzeltmeli Mann-Whitney U testi
#     - Test: gruplar arası ortalama |SHAP| farkı istatistiksel
#             olarak anlamlı mı?
#     - Karşılaştırma: 4 özellik × 3 grup çifti = 12 test
#     - Bonferroni eşiği: α = 0.05 / 12 = 0.0042

N_BOOT       = 1000
CI_LEVEL     = 95
TOP_FEATS    = ['sigma_a','tau_a','phase_proxy','Sy_MPa','E_GPa','UTS_MPa','nu']
TOP_IDX      = [ALL_FEAT.index(f) for f in TOP_FEATS]
TEST_FEATS   = ['Sy_MPa','E_GPa','UTS_MPa','nu']
LP_LIST      = ['Uniaxial','Proportional','Nonproportional']
LP_COLORS    = {'Uniaxial':'#1E88E5','Proportional':'#43A047','Nonproportional':'#E53935'}
N_COMPARISONS = len(TEST_FEATS) * 3
ALPHA_BONF   = 0.05 / N_COMPARISONS


def bootstrap_shap(shap_mat, n_boot=N_BOOT, ci=CI_LEVEL, seed=RANDOM_STATE):
    """
    Bootstrap ile SHAP önem değerleri için yüzdelik güven aralığı.

    Parametreler
    ------------
    shap_mat : (n_samples, n_features) SHAP değer matrisi
    n_boot   : bootstrap tekrar sayısı (varsayılan: 1000)
    ci       : güven düzeyi % (varsayılan: 95)

    Döndürür
    --------
    mean, lower, upper : her biri (n_features,) şeklinde ndarray
    """
    n   = shap_mat.shape[0]
    rng = np.random.default_rng(seed)
    boot = np.array([
        np.abs(shap_mat[rng.integers(0, n, size=n)]).mean(axis=0)
        for _ in range(n_boot)
    ])
    mean  = np.abs(shap_mat).mean(axis=0)
    lower = np.percentile(boot, (100-ci)/2,       axis=0)
    upper = np.percentile(boot, 100-(100-ci)/2,   axis=0)
    return mean, lower, upper


# ── [A] Bootstrap CI grafik ────────────────────────────────────────
fig, axes  = plt.subplots(1, 3, figsize=(18, 6))
ci_results = {}
shap_by_lp = {}

for i, lp in enumerate(LP_LIST):
    mask      = (lp_test == lp)
    n         = mask.sum()
    shap_lp   = shap_values[mask][:, TOP_IDX]
    shap_by_lp[lp] = shap_values[mask]          # tüm özellikler — test B için

    print(f"{lp} (n={n}): bootstrap hesaplanıyor…")
    mean, lower, upper = bootstrap_shap(shap_lp)
    ci_results[lp] = {'feature':TOP_FEATS,'mean':mean,'lower':lower,'upper':upper}

    order    = np.argsort(mean)
    feat_ord = [TOP_FEATS[j] for j in order]
    mean_ord = mean[order]

    ax = axes[i]
    bars = ax.barh(feat_ord, mean_ord, color=LP_COLORS[lp], alpha=0.75, height=0.55)
    ax.errorbar(mean_ord, feat_ord,
                xerr=[mean_ord - lower[order], upper[order] - mean_ord],
                fmt='none', color='#222222', capsize=4, linewidth=1.5)
    ax.set_title(f'{lp}\n(n={n}, %{CI_LEVEL} Bootstrap CI)', fontsize=11)
    ax.set_xlabel('Ortalama |SHAP|', fontsize=10)
    for bar, val in zip(bars, mean_ord):
        ax.text(val+0.003, bar.get_y()+bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)

plt.suptitle(f'LP Bazında SHAP Önem Değerleri (%{CI_LEVEL} Bootstrap CI, n_boot={N_BOOT})',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT,'14_shap_lp_bootstrap_ci.png'), bbox_inches='tight', dpi=150)
plt.show()

# CI tablosunu kaydet
rows = []
for lp in LP_LIST:
    d = ci_results[lp]
    for feat, m, lo, hi in zip(d['feature'], d['mean'], d['lower'], d['upper']):
        rows.append({'LP':lp,'Feature':feat,
                     'Mean':round(float(m),4),
                     'CI_lower_95':round(float(lo),4),
                     'CI_upper_95':round(float(hi),4)})
pd.DataFrame(rows).to_csv(os.path.join(OUT,'15_shap_bootstrap_ci.csv'), index=False)

print(f"\n── Sy_MPa %{CI_LEVEL} Bootstrap CI Özeti ──────────────────────────")
for lp in LP_LIST:
    d   = ci_results[lp]
    idx = list(d['feature']).index('Sy_MPa')
    print(f"  {lp:<22}: {d['mean'][idx]:.4f}  "
          f"[CI: {d['lower'][idx]:.4f} – {d['upper'][idx]:.4f}]")

# ── [B] Bonferroni düzeltmeli Mann-Whitney U Testi ─────────────────
print(f"\n── Bonferroni Düzeltmeli Mann-Whitney U Testi ────────────────")
print(f"   Toplam karşılaştırma: {N_COMPARISONS}  →  α_bonf = {ALPHA_BONF:.4f}")
print(f"\n{'Özellik':<12} {'Uni vs Pro':>16} {'Uni vs Non':>16} {'Pro vs Non':>16}")
print("─" * 64)

mw_rows = []
for feat in TEST_FEATS:
    fidx = ALL_FEAT.index(feat)
    groups = {lp: np.abs(shap_by_lp[lp][:, fidx]) for lp in LP_LIST}

    _, p_up = mannwhitneyu(groups['Uniaxial'],      groups['Proportional'],    alternative='two-sided')
    _, p_un = mannwhitneyu(groups['Uniaxial'],      groups['Nonproportional'], alternative='two-sided')
    _, p_pn = mannwhitneyu(groups['Proportional'],  groups['Nonproportional'], alternative='two-sided')

    def fmt(p):
        return f"p={p:.4f} {'*' if p < ALPHA_BONF else 'ns'}"

    print(f"{feat:<12} {fmt(p_up):>16} {fmt(p_un):>16} {fmt(p_pn):>16}")
    mw_rows.append({'Feature':feat,
                    'p_Uni_vs_Pro':round(p_up,4), 'sig_Uni_vs_Pro': p_up < ALPHA_BONF,
                    'p_Uni_vs_Non':round(p_un,4), 'sig_Uni_vs_Non': p_un < ALPHA_BONF,
                    'p_Pro_vs_Non':round(p_pn,4), 'sig_Pro_vs_Non': p_pn < ALPHA_BONF,
                    'alpha_bonferroni':round(ALPHA_BONF,4)})

print(f"\n  ns = p ≥ {ALPHA_BONF:.4f}  |  * = p < {ALPHA_BONF:.4f} (Bonferroni)")
mw_df = pd.DataFrame(mw_rows)
mw_df.to_csv(os.path.join(OUT,'16_mannwhitney_results.csv'), index=False)
print("✓ Kaydedildi: 14_shap_lp_bootstrap_ci.png, 15_shap_bootstrap_ci.csv, 16_mannwhitney_results.csv")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 11 — SHAP Etkileşim Grafikleri  (Şekil 6)                ║
# ╚══════════════════════════════════════════════════════════════════╝

sy_idx = feature_names.index('Sy_MPa')
e_idx  = feature_names.index('E_GPa')
lp_idx = feature_names.index('lp_encoded')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (fidx, xlabel, title) in zip(axes, [
    (sy_idx, 'σy (MPa)', 'σy × Yükleme Yolu Etkileşimi'),
    (e_idx,  'E (GPa)',  'E × Yükleme Yolu Etkileşimi'),
]):
    sc = ax.scatter(X_test[:, fidx], shap_values[:, fidx],
                    c=X_test[:, lp_idx], cmap='RdYlBu', alpha=0.6, s=25)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_ticks([0, 1, 2])
    cbar.set_ticklabels(['Nonproportional','Proportional','Uniaxial'])
    ax.axhline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('SHAP değeri', fontsize=11)
    ax.set_title(title, fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUT,'09_shap_interaksiyon.png'), bbox_inches='tight', dpi=150)
plt.show()
print("✓ Kaydedildi: 09_shap_interaksiyon.png")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  HÜCRE 12 — Çalışma Sonuç Özeti                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

print("=" * 65)
print("ÇALIŞMA SONUÇ ÖZETİ")
print("=" * 65)

print(f"\n[1] Model Performansı (stratified test seti)")
print(f"    XGBoost  →  R²={r2_xgb:.4f}  RMSE={rmse_xgb:.4f}  ±3x={sb3_xgb:.1f}%")
print(f"    SWT      →  R²={r2_swt:.4f}  RMSE={rmse_swt:.4f}  ±3x={sb3_swt:.1f}%")

print(f"\n[2] 5×10 Tekrarlı K-Fold Kararlılığı")
print(f"    R² = {rkf_scores.mean():.4f} ± {rkf_scores.std():.4f}")

print(f"\n[3] Global SHAP — İlk 5 Özellik")
for _, row in importance_df.head(5).iterrows():
    print(f"    {int(row['rank'])}. {row['feature']:<15} {row['mean_abs_shap']:.4f}")

print(f"\n[4] Mann-Whitney (Bonferroni α={ALPHA_BONF:.4f})")
any_sig = mw_df[['sig_Uni_vs_Pro','sig_Uni_vs_Non','sig_Pro_vs_Non']].any().any()
if any_sig:
    print("    ⚠  Anlamlı fark bulundu — makale metnini güncelleyin!")
    sig_mask = mw_df[['sig_Uni_vs_Pro','sig_Uni_vs_Non','sig_Pro_vs_Non']].any(axis=1)
    print(mw_df[sig_mask].to_string(index=False))
else:
    print("    ✓  Hiçbir özellik için gruplar arası anlamlı fark yok")
    print("       (tüm p > α_bonf — makale Bölüm 3.4 ifadesi desteklenmektedir)")

print(f"\n[5] Üretilen dosyalar → {OUT}")
for f in sorted(os.listdir(OUT)):
    kb = os.path.getsize(os.path.join(OUT, f)) / 1024
    print(f"    {f:<48} {kb:>6.1f} KB")
